# 실전 연습 — 배달 리뷰 데이터 분석

`dataScience/data/delivery_reviews.csv`(30건)로 [분석 나침반](https://claude.ai/code/artifact/94cb6a2a-e51b-42ad-bb72-31d55d1b44ac)의 표준 진행 순서(①~⑦)를 그대로 밟아본다.

**타겟처럼 쓸 변수**: `별점`이 원래는 수치형이지만, Titanic의 `Survived`처럼 이진(binary) 타겟을 연습해보려고 `별점 >= 4.0`이면 `만족`, 아니면 `불만족`으로 새 컬럼을 만든다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats

mpl.rc('font', family='AppleGothic')   # 이 맥에선 AppleGothic (Malgun Gothic 아님)
mpl.rc('axes', unicode_minus=False)

## ① 구조부터 파악한다

In [ ]:
df = pd.read_csv('data/delivery_reviews.csv')

print(df.shape)
df.head()

In [ ]:
df.info()
df.dtypes

## ② 결측치·이상치를 스캔한다

In [ ]:
print(df.isnull().sum())
print()
df.describe()   # min/max가 상식 밖인 컬럼이 있는지 확인

## ③ 타겟변수 자체를 먼저 본다

`별점 >= 4.0` → `만족`, 아니면 `불만족`으로 새 컬럼을 만들고, 그 비율부터 확인 — 이 비율이 이후 모든 그룹 비교의 기준선이 된다.

In [ ]:
df['만족여부'] = np.where(df['별점'] >= 4.0, '만족', '불만족')

counts = df['만족여부'].value_counts()
print(counts)
print(f"\n전체 만족 비율: {counts['만족'] / len(df) * 100:.1f}%")

counts.plot(kind='bar', color=['seagreen', 'salmon'], title='만족여부 비율')
plt.ylabel('건수')
plt.xticks(rotation=0)
plt.show()

## ④ feature 하나하나를 타겟과 교차한다 (반복 작업)

메뉴명(범주형)은 crosstab으로, 가격(수치형)은 그룹별 평균 + boxplot으로 — Titanic 노트북의 `bar_chart(feature)`와 정확히 같은 질문.

In [ ]:
# 범주형 feature(메뉴명) vs 타겟(만족여부) — 비율로 보기
menu_vs_target = pd.crosstab(df['메뉴명'], df['만족여부'], normalize='index').round(2)
print(menu_vs_target.sort_values('만족', ascending=False))

menu_vs_target.plot(kind='bar', stacked=True, figsize=(9, 5), color=['salmon', 'seagreen'])
plt.title('메뉴별 만족/불만족 비율')
plt.ylabel('비율')
plt.xticks(rotation=30)
plt.legend(title='만족여부')
plt.show()

In [ ]:
# 수치형 feature(가격) vs 타겟(만족여부) — 그룹별 평균 + 분포
print(df.groupby('만족여부')['가격'].agg(평균가격='mean', 건수='count').round(0))

plt.figure(figsize=(6, 5))
sns.boxplot(data=df, x='만족여부', y='가격')
plt.title('만족여부별 가격 분포')
plt.show()

## ⑤ feature끼리도 서로 본다

가격과 별점이 관계있는지 — 상관계수 + 산점도.

In [ ]:
corr = df[['가격', '별점']].corr()
print(corr)

plt.figure(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('가격-별점 상관관계')
plt.show()

plt.figure(figsize=(6, 5))
plt.scatter(df['가격'], df['별점'], alpha=0.6)
plt.xlabel('가격')
plt.ylabel('별점')
plt.title('가격 vs 별점 산점도')
plt.show()

## ⑥ 확실히 하고 싶은 차이는 검정으로 뒷받침한다

"만족 그룹과 불만족 그룹의 가격 차이가 우연이 아니다"라고 말할 수 있는지 t-검정으로 확인.

In [ ]:
satisfied_price = df[df['만족여부'] == '만족']['가격']
unsatisfied_price = df[df['만족여부'] == '불만족']['가격']

t_stat, p_value = stats.ttest_ind(satisfied_price, unsatisfied_price, equal_var=False)

print(f"만족 그룹 평균가격: {satisfied_price.mean():.0f}원 (n={len(satisfied_price)})")
print(f"불만족 그룹 평균가격: {unsatisfied_price.mean():.0f}원 (n={len(unsatisfied_price)})")
print(f"\nt-통계량: {t_stat:.3f}, p-value: {p_value:.4f}")

if p_value < 0.05:
    print("-> 두 그룹의 가격 차이는 통계적으로 유의미하다 (p < 0.05)")
else:
    print("-> 두 그룹의 가격 차이가 유의미하다고 보기 어렵다 (p >= 0.05)")

## ⑦ 숫자를 문장으로 번역한다

위 결과들을 보고 직접 결론을 채워보기 — 이게 마지막 단계다.

- 가장 만족도가 높은 메뉴는 ( )이고, 가장 낮은 메뉴는 ( )이다.
- 가격과 만족여부는 관계가 ( 있다 / 없다 ) — 근거: ( )
- 가격과 별점의 상관계수는 ( )로, ( 강한 / 중간 / 약한 ) 상관관계다.
- 만족 그룹과 불만족 그룹의 가격 차이는 통계적으로 ( 유의미하다 / 유의미하지 않다 ).